Objective:
study how people engage with the topic of "hostile architecture"; who drives the conversation, how influence flows between users, and what the sentimental/emotional tone of their contributions reveals. Ultimately, the combination of network and sentiment analysis, will allow for the detection of both structural dynamics and the shared sentiment across communities.

## **Data Collection**

In [ ]:
!pip install praw

!pip install vaderSentiment

!pip install NRCLex
!python -m textblob.download_corpora

In [ ]:
import praw
import requests
import pandas as pd
import json
from datetime import datetime
import getpass
import numpy as np
import seaborn as sns
from collections import Counter
import re
import random

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
api_reddit ={
    "CLIENT_ID" : '',
    "SECRET_TOKEN" : '',
    "USERNAME": '',
    "PASSWORD": ''
}

with open("/content/drive/MyDrive/WSA/Exam_25/redditCredentials.json", "w") as f:
    json.dump(api_reddit, f)

In [ ]:
with open("/content/drive/MyDrive/WSA/Exam_25/redditCredentials.json", "r") as f:
    credentials = json.load(f)

In [ ]:
reddit = praw.Reddit(client_id=credentials['CLIENT_ID'],
                     client_secret=credentials['SECRET_TOKEN'],
                     password=credentials['PASSWORD'],
                     user_agent='Hostile Architecture by /user/No_Explanation_9888',
                     username=credentials['USERNAME'],
                     check_for_async=False)

In [ ]:
print("Logged in as:", reddit.user.me())

## **Posts collection**

In [ ]:
srname = 'HostileArchitecture'
sr = reddit.subreddit(srname)

In [ ]:
submission_generator = sr.new(limit=1000)

submissions = list(submission_generator)

In [ ]:
posts_data = []
for submission in submissions:
    posts_data.append({
        'id': submission.id,
        'title': submission.title,
        'author': submission.author,
        'score': submission.score,
        'comments': submission.comments,
        'num_comments': submission.num_comments,
        'created_utc': submission.created_utc,
        'url': submission.url,
        'selftext': submission.selftext,
        'upvote_ratio': submission.upvote_ratio
    })

In [ ]:
posts_df = pd.DataFrame(posts_data)

# **Comments collection**

In [ ]:
from praw.models import MoreComments

In [ ]:
comments_data = []

for submission in submissions:
    submission.comments.replace_more(limit=None, threshold=10)

    for comment in submission.comments.list():
        if isinstance(comment, MoreComments):
            continue
        comments_data.append({
            'post_id': submission.id,
            'comment_id': comment.id,
            'author': str(comment.author),
            'body': comment.body,
            'created_utc': comment.created_utc,
            'parent_id': comment.parent_id,
            'score': comment.score
        })

In [ ]:
comments_df = pd.DataFrame(comments_data)

In [ ]:
print(f"Total comments collected: {len(comments_df)}")

In [ ]:
comments_df

# **Users collection**

In [ ]:
comment_counts = comments_df.groupby('author').agg(
    comment_count=pd.NamedAgg(column='comment_id', aggfunc='count'),
    avg_score=pd.NamedAgg(column='score', aggfunc='mean')
  ).reset_index()

authors_df = comment_counts.sort_values(by='comment_count', ascending=False) # sort by number of comments

# Function defined for influence_score
authors_df['influence_score'] = (
    0.5 * authors_df['comment_count'] +
    0.5 * authors_df['avg_score']
)

top_users = authors_df.sort_values(by='influence_score', ascending=False)

print(f'Total number of authors collected: {len(authors_df)}')

In [ ]:
authors_df

Why are some authors named "None"? Most likely, these authors' accounts have been either deleted or sunspended (banned). We reasonably believe that None groups together all of the banned/inactive accounts, that's why "the author None" has so many comments.

Does it make sense that JoshuaPearce has made many more comments with respect to the rest of the collected users? Yes, because JoshuaPearce is one of the moderators.

# **Comments preprocessing**

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import nltk
from nltk import FreqDist
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download
nltk.download('omw-1.4')
nltk.download('wordnet')
nltk.download('stopwords')
from nltk.tokenize import TweetTokenizer

In [ ]:
# Initialize tools
tokenizer = TweetTokenizer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# 1. Clean raw text
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http[s]?://\S+', '', text)     # Remove URLs
    text = re.sub(r'<.*?>', '', text)              # Remove HTML
    text = re.sub(r"\\", "", text)                 # Remove backslashes
    text = re.sub(r'\d+', '', text)                # Remove numbers
    text = re.sub(r"\s+", " ", text)               # Normalize spaces
    return text.strip()

# 2. Remove punctuation
def remove_punctuation(text):
    return re.sub(r'[^\w\s]', '', text)

# 3. Tokenize text
def tokenize(text):
    return tokenizer.tokenize(text)

# 4. Remove stopwords
def remove_stopwords(words):
    return [word for word in words if word not in stop_words]

# 5. Lemmatize tokens
def lemmatize(words):
    return [lemmatizer.lemmatize(word) for word in words]


In [ ]:
def preprocess_text_pipeline(text):
    text = clean_text(text)
    text = remove_punctuation(text)
    words = tokenize(text)
    words = remove_stopwords(words)
    words = lemmatize(words)
    return words

In [ ]:
comments_df['processed_body'] = comments_df['body'].astype(str).apply(preprocess_text_pipeline)

In [ ]:
comments_df['processed_body']

# **Comments WordCloud**

In [ ]:
all_words = comments_df['processed_body'].explode().tolist()

In [ ]:
fdist= FreqDist(all_words)

In [ ]:
# Manually define unecessary words to be removed
custom_stopwords = {'hostile', 'architecture', 'homeless', 'dont', 'doesnt', 'would', 'like', 'could', 'theyre', 'youre', 'cant', 'im', 'thats', 'isnt', 'also', 'get'}

# Filter the frequency distribution
filtered_fdist = {word: freq for word, freq in fdist.items() if word not in custom_stopwords}

In [ ]:
wc= WordCloud(width=800, height=400, max_words=50).generate_from_frequencies(filtered_fdist)
plt.imshow(wc, interpolation="bilinear")
plt.show()

This graph is not really infomative.

# **Named Entity Recognition (NER)**
We want to perform NER to extract most mentioned places, organisations, facilities.  

In [ ]:
import spacy
import en_core_web_sm

In [ ]:
nlp = en_core_web_sm.load()

In [ ]:
# Extraction function:
def extract_named_entities(text):
    doc = nlp(text)
    entities = [(ent.text, ent.label_) for ent in doc.ents if ent.label_ in ['GPE', 'ORG', 'FAC']]
    return entities

comments_df['entities'] = comments_df['body'].astype(str).apply(extract_named_entities)

In [ ]:
non_empty_entity_count = comments_df['entities'].apply(lambda x: len(x) > 0 if isinstance(x, list) else False).sum()
print(f"There are {non_empty_entity_count} out of {len(comments_df)} comments with at least one entity")

In [ ]:
# Filter for comments where the 'entities' column is a non-empty list
def print_comments_with_entities(df, num_samples=5):
    has_entities = df[df['entities'].apply(lambda x: isinstance(x, list) and len(x) > 0)]

    print(f"Found {len(has_entities)} comments with at least one entity.\n")

    for idx, row in has_entities.sample(n=min(num_samples, len(has_entities)), random_state=1).iterrows():
        print(f"Comment ID: {row['comment_id']}")
        print(f"Author: {row['author']}")
        print("Entities:", row['entities'])
        print("Comment:", row['body'])
        print("-" * 80)

In [ ]:
print_comments_with_entities(comments_df, num_samples=5)

Now that we have extracted the most mentioned entities, we can visualise the mentioning frequency, so that we can have an idea of hotspots, most discussed people and organisations.

**Analysis of entities (geographical locations, organizations, facilities)**

In [ ]:
all_entities = comments_df['entities'].explode().dropna()
entity_freq = Counter(all_entities)
top_entities = entity_freq.most_common(50)

top_entities_df = pd.DataFrame(top_entities, columns=['Entity', 'Count'])

In [ ]:
# Extract entity labels (e.g., 'GPE', 'ORG', 'FAC') from the entities column
def extract_entity_types(entity_list):
    return [label for (_, label) in entity_list if label in ['GPE', 'ORG', 'FAC']]

comments_df['entity_labels'] = comments_df['entities'].apply(extract_entity_types)

# Flatten all entity labels into one list
all_labels = comments_df['entity_labels'].explode().dropna()

# Count frequencies
label_counts = Counter(all_labels)
label_names = list(label_counts.keys())
label_values = list(label_counts.values())

# Pie chart
plt.figure(figsize=(3, 3))
plt.pie(label_values, labels=label_names, autopct='%1.1f%%', startangle=140, colors=['#32cd32','#ffd700','#00bfff'])
plt.title('Distribution of Named Entity Types (GPE, ORG, FAC)', pad=25)
plt.axis('equal')
plt.tight_layout()
plt.show()

In [ ]:
gpe_counter = Counter()
org_counter = Counter()
fac_counter = Counter()

for row in comments_df['entities']:
    for name, label in row:
        if label == 'GPE':
            gpe_counter[name] += 1
        elif label == 'ORG':
            org_counter[name] += 1
        else:
            fac_counter[name] += 1

top_gpe = pd.DataFrame(gpe_counter.most_common(10), columns=['Entity', 'Count'])
top_org = pd.DataFrame(org_counter.most_common(10), columns=['Entity', 'Count'])
top_fac = pd.DataFrame(fac_counter.most_common(10), columns=['Entity', 'Count'])

In [ ]:
#Side-by-side plots
fig, axs = plt.subplots(1, 3, figsize=(12, 4))

#GPE
axs[0].barh(top_gpe['Entity'], top_gpe['Count'], color="#32cd32")
axs[0].set_title("Top 10 GPE Entities")
axs[0].set_xlabel("Mentions")
axs[0].invert_yaxis()

#ORG
axs[1].barh(top_org['Entity'], top_org['Count'], color="#ffd700")
axs[1].set_title("Top 10 ORG Entities")
axs[1].set_xlabel("Mentions")
axs[1].invert_yaxis()

#FAC
axs[2].barh(top_fac['Entity'], top_fac['Count'], color="#00bfff")
axs[2].set_title("Top 10 FAC Entities")
axs[2].set_xlabel("Mentions")
axs[2].invert_yaxis()

plt.tight_layout()
plt.show()

Now that we know which places are the most discussed, we would like to analyse the frequency of urban-related words  (commonly expression of hostile architecture) and their co-occurrence with geo-political entities.

Since our pipeline en_core_web_sm doesn't recognize them, we decided to manually group what we believe to be the most common keywords in this field (we have also surfed the subreddit and the net to find more).

In [ ]:
urban_keywords = [
    "bench",
    "pavement",
    "bus",
    "bus stop",
    "sidewalk",
    "curb",
    "station",
    "platform",
    "crosswalk",
    "spikes",
    "seat",
    "bike rack",
    "armrests",
    "parking",
    "trash can",
    "mailbox",
    "desk",
    "park",
    "bar"
]

In [ ]:
#Initialize counter
keyword_counts = Counter()

#Count occurrences of each keyword in the body column
for keyword in urban_keywords:
    pattern = r'\b' + re.escape(keyword) + r'\b'
    count = comments_df['body'].str.count(pattern).sum()
    keyword_counts[keyword] = count

sorted_keywords = sorted(keyword_counts.items(), key=lambda x: x[1])

keywords, counts = zip(*sorted_keywords)
plt.figure(figsize=(8, 5))
plt.barh(keywords, counts)
plt.xlabel("Frequency")
plt.title("Occurrences of Urban Keywords in Comments")
plt.tight_layout()
plt.show()

In [ ]:
#Top 5 urban keywords
top_urban_words = [kw for kw, _ in sorted(keyword_counts.items(), key=lambda x: x[1], reverse=True)[:5]]

#Top 5 GPEs
top_gpes = [gpe for gpe, _ in gpe_counter.most_common(5)]

In [ ]:
#Empty DataFrame to store counts
co_matrix = pd.DataFrame(0, index=top_gpes, columns=top_urban_words)

for _, row in comments_df.iterrows():
    if not isinstance(row['entities'], list) or not isinstance(row['body'], str):
        continue

    comment_text = row['body'].lower()
    present_keywords = [word for word in top_urban_words if re.search(r'\b' + re.escape(word) + r'\b', comment_text)]
    present_gpes = [ent[0] for ent in row['entities'] if ent[1] == 'GPE' and ent[0] in top_gpes]

    for gpe in present_gpes:
        for word in present_keywords:
            co_matrix.loc[gpe, word] += 1

In [ ]:
plt.figure(figsize=(6, 4.5))
sns.heatmap(co_matrix, annot=True, cmap="YlOrRd", fmt="d")

plt.title("Co-occurrence of Top GPEs and Urban Keywords", pad=20)
plt.xlabel("Urban Keyword", labelpad=15)
plt.ylabel("GPE", labelpad=15)

plt.tight_layout()
plt.show()

# **General sentiment analysis on comments (VADER)**
We chose VADER among all the other sentiment analysis tools, as it is the best for analysing sentiments expressed in social media.

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [ ]:
#Initialize VADER
analyzer = SentimentIntensityAnalyzer()

#Classify sentiment
def classify_sentiment(text):
    sid_obj = SentimentIntensityAnalyzer()

    sentiment_dict = sid_obj.polarity_scores(text)
    compound = sentiment_dict['compound']

    if compound > 0.05:
        return 'positive'
    elif compound < -0.05:
        return 'negative'
    else:
        return 'neutral'

comments_df['sentiment'] = comments_df['body'].apply(classify_sentiment)

We are applying VADER to not preprocessed body of the comments dataframe, as caplocks, punctuation, degree modifiers... are fundamental to precisely determine the tone.

In [ ]:
#Show distribution of sentiment in the collected comments
sentiment_counts = comments_df['sentiment'].value_counts()
print(sentiment_counts)

In [ ]:
plt.figure(figsize=(4, 4))
sentiment_counts.plot(kind='pie', autopct='%1.1f%%', colors=['#F2D82F', '#FC69D0', '#49ADF9'])
plt.title('Sentiment Distribution')
plt.ylabel('')
plt.show()

In [ ]:
# One example for each sentiment
example_positive = comments_df[comments_df['sentiment'] == 'positive'].sample(1)['body'].values[0]
example_negative = comments_df[comments_df['sentiment'] == 'negative'].sample(1)['body'].values[0]
example_neutral = comments_df[comments_df['sentiment'] == 'neutral'].sample(1)['body'].values[0]

print("Positive Comment:\n", example_positive)
print("\nNegative Comment:\n", example_negative)
print("\nNeutral Comment:\n", example_neutral)

Interpretation: does positive mean that these comments support hostile architecture? No, positive sentiment might be: sarcastic praise (ex: "Great, now we’re putting spikes on benches too. What a brilliant idea."), approval of resistance (ex: "Love that people are calling this out!"), or commending design cleverness (ex: "It’s actually a smart way to prevent people from loitering.").

## **Emotions Detection**

In [ ]:
from nrclex import NRCLex

In [ ]:
def detect_emotions(text):
    emotion = NRCLex(text)
    return emotion.raw_emotion_scores

In [ ]:
comments_df['emotions'] = comments_df['body'].apply(detect_emotions)

In [ ]:
all_emotions = [emotion for sublist in comments_df['emotions'] for emotion in sublist]

#Count frequency of each emotion
emotion_counts = Counter(all_emotions)

emotion_df = pd.DataFrame(emotion_counts.items(), columns=['Emotion', 'Count'])
emotion_df = emotion_df.sort_values(by='Count', ascending=False)

plt.figure(figsize=(6, 3))
plt.bar(emotion_df['Emotion'], emotion_df['Count'], color='coral')
plt.xlabel('Emotion')
plt.ylabel('Frequency')
plt.title('Overall Emotion Distribution in Comments')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# **Emotion Analysis on users**

 Including "None" (deleted or banned accounts) skews our analysis of active users, so we decide to filter them out.

In [ ]:
#Filter out None or "None" authors
filtered_comments_df = comments_df[comments_df['author'].notna() & (comments_df['author'] != "None")]

In [ ]:
#Count how many comments each author made
user_counts = filtered_comments_df['author'].value_counts()

user_counts.head(10).plot(kind='bar', figsize=(7, 4), color='purple')
plt.title('Top 10 Most Active Users')
plt.xlabel('User')
plt.ylabel('Number of Comments')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Get top 10 most active users and filter their comments
top_users = filtered_comments_df['author'].value_counts().nlargest(10).index.tolist()
top_comments = filtered_comments_df[filtered_comments_df['author'].isin(top_users)].copy()

# Expand 'emotions' dictionary to columns and add 'author' column
emotion_expanded = top_comments['emotions'].apply(pd.Series)

emotion_expanded['author'] = top_comments['author'].values

emotion_counts = emotion_expanded.groupby('author').sum()

plt.figure(figsize=(11, 6))
for user in emotion_counts.index:
    plt.plot(emotion_counts.columns, emotion_counts.loc[user], label=user, marker='o')

plt.title('Emotion Distribution for Top 10 Active Users')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.legend(title='User', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

# **Sentiment and Emotional Analysis on posts**

In [ ]:
# Combine title and selftext into one full text column
posts_df['full_text'] = posts_df['title'].astype(str) + " " + posts_df['selftext'].astype(str)

posts_df['compound'] = posts_df['full_text'].apply(lambda x: analyzer.polarity_scores(x)['compound'])
posts_df['sentiment'] = posts_df['full_text'].apply(classify_sentiment)

In [ ]:
cols = ['title', 'compound', 'score', 'num_comments']

# Sort by compound sentiment score
top_positive_posts = posts_df.sort_values(by='compound', ascending=False).head(5)
top_negative_posts = posts_df.sort_values(by='compound').head(5)


# Truncate long titles for cleaner display
top_positive_display = top_positive_posts[cols].copy()
top_positive_display['title'] = top_positive_display['title']

top_negative_display = top_negative_posts[cols].copy()
top_negative_display['title'] = top_negative_display['title']

print("=== Top 5 Positive Posts ===\n")

for idx, row in top_positive_display.reset_index(drop=True).iterrows():
    print(f"{idx + 1}. {row['title']}")
    print(f"   Sentiment Score: {row['compound']}")
    print(f"   Upvotes: {row['score']} | Comments: {row['num_comments']}\n")

print("\n=== Top 5 Negative Posts ===\n")

for idx, row in top_negative_display.reset_index(drop=True).iterrows():
    print(f"{idx + 1}. {row['title']}")
    print(f"   Sentiment Score: {row['compound']}")
    print(f"   Upvotes: {row['score']} | Comments: {row['num_comments']}\n")

What kind of emotions do negative posts evoke with respect to positive posts? We compare the emotions expressed in the top 10 positive and negative posts.

In [ ]:
# Retrieval of 10 posts per sentiment
top_positive = posts_df.sort_values(by='compound', ascending=False).head(10)
top_negative = posts_df.sort_values(by='compound').head(10)

# Count emotions per post in each group (positive/negative)
def aggregate_emotions(df):
    emotion_counter = {}
    for text in df['full_text']:
        emotions = NRCLex(text).raw_emotion_scores
        for emotion, count in emotions.items():
            emotion_counter[emotion] = emotion_counter.get(emotion, 0) + count
    return emotion_counter

pos_emotions = aggregate_emotions(top_positive)
neg_emotions = aggregate_emotions(top_negative)

all_emotions = sorted(set(pos_emotions.keys()).union(set(neg_emotions.keys())))
data = {
    'emotion': all_emotions,
    'positive': [pos_emotions.get(e, 0) for e in all_emotions],
    'negative': [neg_emotions.get(e, 0) for e in all_emotions]
}
emotion_df = pd.DataFrame(data)

# Remove 'positive' and 'negative' from the emotion list
filtered_emotions = [e for e in all_emotions if e not in ['positive', 'negative']]

data = {
    'emotion': filtered_emotions,
    'positive': [pos_emotions.get(e, 0) for e in filtered_emotions],
    'negative': [neg_emotions.get(e, 0) for e in filtered_emotions]
}
emotion_df = pd.DataFrame(data)


x = range(len(emotion_df))
bar_width = 0.35

plt.figure(figsize=(7, 4))
plt.bar(x, emotion_df['positive'], width=bar_width, label='Positive Posts', color='#C9CC3F')
plt.bar([i + bar_width for i in x], emotion_df['negative'], width=bar_width, label='Negative Posts', color='#E0115F')

plt.xticks(ticks=[i + bar_width / 2 for i in x], labels=emotion_df['emotion'], rotation=45)
plt.xlabel("Emotion")
plt.ylabel("Frequency", labelpad=10)
plt.title("Emotion Comparison: top 10 positive vs top 10 negative posts")
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

As we can see from the plot above, there are significant differences among the emotions conveyed by negative and positive posts.

In particular, positive posts are correctly associated to typically positive emotions: joy, anticipation, trust... Instead, negative posts tend to be more associated with anger, disgust, fear, sadness...

# **Network Analysis**

In [ ]:
import networkx as nx

In [ ]:
# Initialize directed graph
G = nx.DiGraph()

# Preprocess - create a dictionary of comment_id and its author
id_to_author = dict(zip(comments_df['comment_id'], comments_df['author']))

# Build the edges
for _, row in comments_df.iterrows():
    source = row['author']
    parent_id = row['parent_id'].split('_')[-1]  #extract parent_id and remove "t1_" or "t3_" prefix
    target = id_to_author.get(parent_id) #find who wrote the parent comment

    # Add edge ONLY if users are valid (not the same and not None)
    if source and target and source != 'None' and target != 'None' and source != target:
        G.add_edge(source, target)

In [ ]:
print(f"Number of users (nodes): {G.number_of_nodes()}")
print(f"Number of reply interactions (edges): {G.number_of_edges()}")

In [ ]:
# Get top 50 users by degree
top_nodes = sorted(G.degree, key=lambda x: x[1], reverse=True)[:50]
top_node_names = [node for node, _ in top_nodes]
subgraph = G.subgraph(top_node_names)

# Identify the top users
top_user = top_node_names[0]
top_users_set = set(top_node_names[:10])  # Top 10 users

# Define node sizes and colors
node_sizes = [
    500 if node == top_user else 300 if node in top_users_set else 100
    for node in subgraph.nodes()
]
node_colors = [
    'fuchsia' if node == top_user else 'orange' if node in top_users_set else 'turquoise'
    for node in subgraph.nodes()
]

plt.figure(figsize=(13, 9))
pos = nx.spring_layout(subgraph, k=0.4, seed=42)

nx.draw_networkx_nodes(subgraph, pos, node_size=node_sizes, node_color=node_colors, alpha=0.9)
nx.draw_networkx_edges(subgraph, pos, arrows=True, alpha=0.3)
nx.draw_networkx_labels(subgraph, pos, font_size=7)

plt.title("Top 50 Users Interaction Subgrap by Degree Centrality", fontsize=16)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Compute centrality measures
degree_centrality = nx.degree_centrality(G)
betweenness_centrality = nx.betweenness_centrality(G)

centrality_df = pd.DataFrame({
    'User': list(G.nodes),
    'Degree Centrality': [degree_centrality[node] for node in G.nodes],
    'Betweenness Centrality': [betweenness_centrality[node] for node in G.nodes]
})

# Sort by Degree Centrality to match the subgraph nodes
centrality_df.sort_values(by='Degree Centrality', ascending=False, inplace=True)

top_10_all = centrality_df.head(10)
top_10_all

In [ ]:
# Get top 50 users by betweenness centrality
top_nodes = sorted(betweenness_centrality.items(), key=lambda x: x[1], reverse=True)[:50]
top_node_names = [node for node, _ in top_nodes]
subgraph = G.subgraph(top_node_names)


# Identify the top users
top_user = top_node_names[0]
top_users_set = set(top_node_names[:10])  # Top 10 users

# Define node sizes and colors
node_sizes = [
    500 if node == top_user else 300 if node in top_users_set else 100
    for node in subgraph.nodes()
]
node_colors = [
    'fuchsia' if node == top_user else 'orange' if node in top_users_set else 'turquoise'
    for node in subgraph.nodes()
]


plt.figure(figsize=(13, 9))
pos = nx.spring_layout(subgraph, k=0.4, seed=42)

nx.draw_networkx_nodes(subgraph, pos, node_size=node_sizes, node_color=node_colors, alpha=0.9)
nx.draw_networkx_edges(subgraph, pos, arrows=True, alpha=0.3)
nx.draw_networkx_labels(subgraph, pos, font_size=7)

plt.title("Top 50 Users Interaction Subgraph by Betweenness Centrality", fontsize=16)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Sort by Betweenness Centrality
centrality_df.sort_values(by='Betweenness Centrality', ascending=False, inplace=True)

top_10_betweenness = centrality_df.head(10)
top_10_betweenness

The two graphs above represent the top 50 users interactions in the comments to make a comparison between actual popularity and structural importance:

1. Degree centrality: shows the most replied-to users most active or most replied-to users (popularity)

2. Betweenness centrality: shows the users who act as bridges between communities (structural importance)

Why are there some isolated users? It might be that they only wrote top-level comments, not replies. And also, if they have no incident edges, then their comments did not get any replies.

# **Temporal Analysis**

If we had access to the growth of new subscribers or to the change in upvotes and downvotes on posts overtime, we could plot some line charts.
However, since PRAW doesn't allow users to retrieve those kind of data, the only way to perform this analysis would be to manually keep a log of how these data change.

In [ ]:
import matplotlib.dates as mdates

In [ ]:
posts_df['created_datetime'] = pd.to_datetime(posts_df['created_utc'], unit='s')
posts_df['month'] = posts_df['created_datetime'].dt.to_period('M')

In [ ]:
# Convert 'created_utc' from UNIX timestamp to datetime
posts_df['created_datetime'] = pd.to_datetime(posts_df['created_utc'], unit='s')

# Extract useful time components for analysis
posts_df['year'] = posts_df['created_datetime'].dt.year
posts_df['month'] = posts_df['created_datetime'].dt.month
posts_df['day'] = posts_df['created_datetime'].dt.day
posts_df['hour'] = posts_df['created_datetime'].dt.hour
posts_df['weekday'] = posts_df['created_datetime'].dt.weekday  # 0 = Monday

print(posts_df[['created_utc', 'created_datetime', 'year', 'month', 'day', 'hour', 'weekday']].head(10))

In [ ]:
# Group by month
monthly_posts = posts_df.resample('ME', on='created_datetime').size()

plt.figure(figsize=(13, 4))
monthly_posts.plot(marker='o', color='fuchsia')
plt.title("Monthly Growth of Posts")
plt.xlabel("Date")
plt.ylabel("Number of Posts")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Resample post counts by year
yearly_posts = posts_df.resample('YE', on='created_datetime').size()

plt.figure(figsize=(10, 3))
yearly_posts.plot(marker='o', color='royalblue')
plt.title("Yearly Growth of Posts")
plt.xlabel("Year")
plt.ylabel("Number of Posts")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

The subreddit was created on 16th may 2016, however, the plot would show no posts growth from 2016 to 2021 beacuse we only retrieved 2000 posts. Thereore, we decided to only show the growth in these past years.

In [ ]:
# Define a color for each sentiment
sentiment_colors = {
    'Positive': 'green',
    'Negative': 'red',
    'Neutral': 'orange'
}

# Group by year and sentiment
yearly_sentiment = posts_df.set_index('created_datetime').groupby('sentiment').resample('YS').size().unstack(level=0).fillna(0)

plt.figure(figsize=(11, 4))
for sentiment in yearly_sentiment.columns:
    plt.plot(
        yearly_sentiment.index,
        yearly_sentiment[sentiment],
        label=sentiment,
        marker='o',
        color=sentiment_colors.get(sentiment)
    )

plt.title("Yearly Growth of Posts by Sentiment")
plt.xlabel("Year")
plt.ylabel("Number of Posts")
plt.grid(True)

date_format = mdates.DateFormatter('%b %Y')
plt.gca().xaxis.set_major_formatter(date_format)
plt.gca().xaxis.set_major_locator(mdates.YearLocator())

plt.xticks(rotation=45)
plt.legend(title='Sentiment', loc='center left', bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

We noticed there is a peak in 2022, and as time went by, the sentiment enagagement has decreased significantly. We see that from 2023 the most common sentiments are negative and neutral, but is it just a trend for the posts, or is it mirrored by the commenters too? (= are active commenters more positive or negative/neutral?)

## **Insight on users' activity levels and sentiments**

In [ ]:
import matplotlib.patches as mpatches
from collections import defaultdict

In [ ]:
# Count number of comments per user
comment_counts = filtered_comments_df['author'].astype(str).value_counts()

# Categorize users
def classify_comment_activity(count):
    if count < 2:
        return 'inactive'
    elif count < 6:
        return 'semi-active'
    else:
        return 'active'

activity_levels = comment_counts.apply(classify_comment_activity)
activity_df = activity_levels.reset_index()
activity_df.columns = ['author', 'activity_level']

# Build interaction graph from comment replies
G = nx.DiGraph()

for _, row in filtered_comments_df.iterrows():
    author = str(row['author'])
    parent_id = row['parent_id']

    if parent_id.startswith("t1_"):  # comment reply
        parent_comment = filtered_comments_df[filtered_comments_df['comment_id'] == parent_id[3:]]
        if not parent_comment.empty:
            parent_author = str(parent_comment.iloc[0]['author'])
            if parent_author != 'None' and author != parent_author:
                G.add_edge(author, parent_author)

# Assign activity levels as node attributes
activity_dict = dict(zip(activity_df['author'], activity_df['activity_level']))
nx.set_node_attributes(G, activity_dict, name='activity_level')

# Filter graph to keep only users in activity_df
G_filtered = G.subgraph(activity_df['author'])

In [ ]:
activity_distribution = activity_levels.value_counts().sort_index()

activity_plot_df = pd.DataFrame({
    'Activity Level': activity_distribution.index,
    'Count': activity_distribution.values
})

color_map = {'inactive': '#FF8000', 'semi-active': '#0080FF', 'active': '#66CC00'}
node_colors = [color_map.get(G_filtered.nodes[n].get('activity_level', 'inactive')) for n in G_filtered.nodes()]

plt.figure(figsize=(6, 4))
sns.barplot(
    data=activity_plot_df,
    x='Activity Level',
    y='Count',
    hue='Activity Level',
    palette=color_map,
    legend=False
)
plt.title("User Activity Levels in Comments")
plt.xlabel("Activity Level")
plt.ylabel("Number of Users")
plt.tight_layout()
plt.show()

In [ ]:
# Ensure authors have sentiment labels and activity levels
valid_sentiment_df = filtered_comments_df[
    filtered_comments_df['author'].notna() &
    filtered_comments_df['sentiment'].notna()
].copy()

# Add activity level per user
valid_sentiment_df['activity_level'] = valid_sentiment_df['author'].map(activity_dict)

# Compute average sentiment per user
user_sentiment = valid_sentiment_df.groupby(['author', 'sentiment']).size().unstack(fill_value=0)

# Find dominant sentiment
user_sentiment['total'] = user_sentiment.sum(axis=1)
user_sentiment['dominant_sentiment'] = user_sentiment[['positive', 'negative', 'neutral']].idxmax(axis=1)

# Merge with activity level
user_sentiment['activity_level'] = user_sentiment.index.map(activity_dict)

# Drop users with unknown activity level
user_sentiment = user_sentiment.dropna(subset=['activity_level'])

# Count dominant sentiment per activity group
summary = user_sentiment.groupby(['activity_level', 'dominant_sentiment']).size().unstack(fill_value=0)

In [ ]:
# Graph
colors = {'positive': '#66CC00', 'negative': '#FF0000', 'neutral': '#66B2FF'}
summary = summary[['positive', 'negative', 'neutral']]  # consistent column order

summary.plot(
    kind='bar',
    figsize=(5, 3),
    color=[colors[c] for c in summary.columns]
)

plt.title("Sentiment Distribution by User Activity Group")
plt.xlabel("User Activity Level")
plt.ylabel("Number of Users")
plt.legend(title="Dominant Sentiment", loc='center left', bbox_to_anchor=(1, 0.5))
plt.show()

In [ ]:
# Extract user's sentiment & activity
user_sentiment_map = user_sentiment['dominant_sentiment'].to_dict()
user_activity_map = user_sentiment['activity_level'].to_dict()

G_clusters = nx.Graph()

# Add nodes grouped by cluster
cluster_positions = {
    'inactive': (-1, 0),
    'semi-active': (0, 0),
    'active': (1, 0)
}

cluster_offset = {
    'inactive': (-1, 0),
    'semi-active': (0, 0),
    'active': (1, 0)
}

# Add nodes and assign positions and colors
positions = {}
node_colors = []
cluster_spacing = 1.0

for user in user_sentiment.index:
    activity = user_activity_map.get(user)
    sentiment = user_sentiment_map.get(user)
    if activity and sentiment:
        G_clusters.add_node(user)
        positions[user] = (
            cluster_offset[activity][0] + np.random.normal(scale=0.1),
            cluster_offset[activity][1] + np.random.normal(scale=0.1)
        )
        node_colors.append(colors[sentiment])

plt.figure(figsize=(9, 6))
nx.draw(
    G_clusters,
    pos=positions,
    node_color=node_colors,
    node_size=80,
    alpha=0.8
)

# Cluster labels
for label, (x, y) in cluster_offset.items():
    plt.text(x, y + 0.3, label.upper(), fontsize=12, ha='center', color='black')

# Legend for sentiments
legend_elements = [
    mpatches.Patch(color=colors['positive'], label='Positive'),
    mpatches.Patch(color=colors['negative'], label='Negative'),
    mpatches.Patch(color=colors['neutral'], label='Neutral')
]
plt.legend(handles=legend_elements, title='Dominant Sentiment', loc='upper right', bbox_to_anchor=(1.3, 0.6))

plt.title("Users Clustered by Activity Level and Colored by Sentiment")
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Counter for sentiments per activity level
cluster_sentiment_counts = defaultdict(Counter)

for user in user_sentiment.index:
    activity = user_activity_map.get(user)
    sentiment = user_sentiment_map.get(user)
    if activity and sentiment:
        cluster_sentiment_counts[activity][sentiment] += 1

for activity_level in ['inactive', 'semi-active', 'active']:
    print(f"\n{activity_level.upper()} users sentiment communities (number of members):")
    for sentiment in ['positive', 'negative', 'neutral']:
        count = cluster_sentiment_counts[activity_level][sentiment]
        print(f"  {sentiment.capitalize()}: {count}")

# **Community Detection**

In [ ]:
from networkx.algorithms.community import greedy_modularity_communities

from math import cos, sin, pi
import matplotlib.cm as cm
import matplotlib.colors as mcolors

In [ ]:
# Map comment_id to author
comment_author_map = dict(zip(filtered_comments_df['comment_id'], filtered_comments_df['author']))

# Build interaction graph (user replies to user)
G = nx.DiGraph()
for _, row in filtered_comments_df.iterrows():
    source = row['author']
    parent_id = str(row['parent_id'])
    target_id = parent_id[3:] if parent_id.startswith(('t1_', 't3_')) else parent_id
    if target_id in comment_author_map:
        target = comment_author_map[target_id]
        if source != target:
            G.add_edge(source, target)

# Convert to undirected and apply Greedy Modularity
G_undirected = G.to_undirected()
communities = list(greedy_modularity_communities(G_undirected))

# Assign community IDs
partition = {}
for i, comm in enumerate(communities):
    for user in comm:
        partition[user] = i

# Group users by community
community_nodes = defaultdict(list)
for user, comm_id in partition.items():
    community_nodes[comm_id].append(user)

# Activity/Sentiment maps
user_activity = filtered_comments_df['author'].value_counts().to_dict()
user_sentiment = (
    filtered_comments_df.groupby('author')['sentiment']
    .agg(lambda x: Counter(x).most_common(1)[0][0])
    .to_dict()
)

# Community-level stats
community_stats = []
for comm_id, users in community_nodes.items():
    activities = [user_activity.get(u, 0) for u in users]
    sentiments = [user_sentiment.get(u, 'unknown') for u in users]
    dominant_sentiment = Counter(sentiments).most_common(1)[0][0]
    top_users = sorted(users, key=lambda u: G_undirected.degree(u), reverse=True)[:3]

    community_stats.append({
        "Community ID": comm_id,
        "Size": len(users),
        "Dominant Sentiment": dominant_sentiment,
        "Top Central Users": ", ".join(top_users)
    })

community_stats_df = pd.DataFrame(community_stats)

# Considering only communities ≥ 50
community_stats_df['Community Label'] = community_stats_df.apply(
    lambda row: f"Community {row['Community ID']}" if row['Size'] >= 50 else "Other", axis=1
)

# Filter communities with Size ≥ 100 (most meaningful by number of members)
filtered_stats_df = community_stats_df[community_stats_df['Size'] >= 100].copy()

display(filtered_stats_df)

In [ ]:
# Get valid community IDs from the filtered DataFrame
valid_community_ids = filtered_stats_df['Community ID'].tolist()

# Community colors (COLOR_MAP)
n_communities = len(valid_community_ids)
colormap = cm.get_cmap('tab20', n_communities)
color_map = {
    comm_id: mcolors.to_hex(colormap(i)) for i, comm_id in enumerate(valid_community_ids)
}

# Community node groups
def get_community_node_groups(node_list):
    community_node_groups = {comm_id: [] for comm_id in valid_community_ids}
    for node in node_list:
        comm_id = partition[node]
        if comm_id in community_node_groups:
            community_node_groups[comm_id].append(node)
    return community_node_groups

def compute_positions(community_node_groups):
    positions = {}
    angle_step = 2 * pi / len(community_node_groups)
    radius = 5
    for i, (comm_id, nodes) in enumerate(community_node_groups.items()):
        angle = i * angle_step
        offset = (cos(angle) * radius, sin(angle) * radius)
        subgraph = G.subgraph(nodes)
        sub_pos = nx.spring_layout(subgraph, seed=42, k=0.5)
        for node, pos_ in sub_pos.items():
            positions[node] = (pos_[0] + offset[0], pos_[1] + offset[1])
    return positions

# Legend handles
legend_handles = [
    mpatches.Patch(color=color_map[cid], label=f'Community {cid}')
    for cid in valid_community_ids
]

In [ ]:
# Graph1: sampled users for visualisation purposes

# Sample nodes
sampled_nodes = []
for comm_id in valid_community_ids:
    users_in_comm = [u for u in partition if partition[u] == comm_id]
    n_sample = max(1, len(users_in_comm) // 5)
    sampled = random.sample(users_in_comm, min(len(users_in_comm), n_sample))
    sampled_nodes.extend(sampled)

# Create subgraph
G_sampled = G.subgraph(sampled_nodes).copy()

community_node_groups_sampled = get_community_node_groups(sampled_nodes)
positions_sampled = compute_positions(community_node_groups_sampled)

node_colors_sampled = [color_map[partition[n]] for n in G_sampled.nodes()]

plt.figure(figsize=(14, 12))
nx.draw_networkx_nodes(G_sampled, positions_sampled, node_color=node_colors_sampled, node_size=150, alpha=0.9)
nx.draw_networkx_edges(G_sampled, positions_sampled, alpha=0.2, width=0.5)
nx.draw_networkx_labels(G_sampled, positions_sampled, font_size=6)

plt.legend(handles=legend_handles, loc='center left', bbox_to_anchor=(1.05, 0.5), title="Communities")
plt.title("Clustered Community Layout (Sampled Users)", fontsize=16)
plt.axis('off')
plt.tight_layout()
plt.show()

Communities with just 2 or 3 users are usually statistically insignificant, as they represent loose or isolated replies... they just add noise.

## **Hubs identifications: top influencers**

In [ ]:
# Graph2: all users display

all_nodes = []
for comm_id in valid_community_ids:
    users_in_comm = [u for u in partition if partition[u] == comm_id]
    all_nodes.extend(users_in_comm)

# Create subgraph
G_full = G.subgraph(all_nodes).copy()

community_node_groups_full = get_community_node_groups(all_nodes)
positions_full = compute_positions(community_node_groups_full)

node_colors_full = [color_map.get(partition[n], 'gray') for n in G_full.nodes()]

plt.figure(figsize=(14, 12))
nx.draw_networkx_nodes(G_full, positions_full, node_color=node_colors_full, node_size=150, alpha=0.9)
nx.draw_networkx_edges(G_full, positions_full, alpha=0.2, width=0.5)
nx.draw_networkx_labels(G_full, positions_full, font_size=6)

plt.legend(handles=legend_handles, loc='center left', bbox_to_anchor=(1.05, 0.5), title="Communities")
plt.title("Clustered Community Layout (All Users in Selected Communities)", fontsize=16)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Graph3: top influencers across all communities

# Compute centrality
degree_centrality = nx.degree_centrality(G_full)
betweenness_centrality = nx.betweenness_centrality(G_full)
closeness_centrality = nx.closeness_centrality(G_full)

# Hybrid influence score
influence_score = {
    node: (
        0.5 * degree_centrality.get(node, 0) +
        0.3 * betweenness_centrality.get(node, 0) +
        0.2 * closeness_centrality.get(node, 0)
    )
    for node in G_full.nodes()
}

# Top influencers
top_influencers = sorted(influence_score.items(), key=lambda x: x[1], reverse=True)[:20]
influencer_nodes = [node for node, _ in top_influencers]
top_influencer_nodes = set(influencer_nodes)
other_nodes = [n for n in G_full.nodes() if n not in top_influencer_nodes]

# Edges involving influencers
influencer_edges = [
    (u, v) for u, v in G_full.edges()
    if u in top_influencer_nodes or v in top_influencer_nodes
]

influencer_colors = ['yellow'] * len(influencer_nodes)
other_colors = [color_map[partition[n]] for n in other_nodes]

plt.figure(figsize=(14, 12))

# Elements of the graph
nx.draw_networkx_edges(G_full, positions_full, edgelist=influencer_edges, alpha=0.3, width=1.0, arrows=True, arrowsize=15)

nx.draw_networkx_nodes(G_full, positions_full, nodelist=other_nodes, node_color=other_colors, node_size=150, alpha=0.9)

nx.draw_networkx_nodes(G_full, positions_full, nodelist=influencer_nodes, node_color=influencer_colors, node_size=300, alpha=1)

influencer_labels = {node: node for node in influencer_nodes}
nx.draw_networkx_labels(G_full, positions_full, labels=influencer_labels, font_size=9, font_weight='bold', font_color='black')

legend_elements_influencers = [
    mpatches.Patch(color='yellow', label='Top Influencers')
] + legend_handles

plt.legend(handles=legend_elements_influencers, loc='center left', bbox_to_anchor=(1.05, 0.5), title="Communities")
plt.title("Clustered Community Layout with Top Influencers Highlighted", fontsize=16)
plt.axis('off')
plt.tight_layout()
plt.show()

As expected, the top influencers are located in Community 0, which is the one which has the largest size (= the most interactions). And these nodes act as a bridge to the other communities.

## **Hubs vs community sentiment**

Do influencers spread a positive sentiment?
How does their sentiment compare to their communities?

In [ ]:
# Build table for influencers vs community sentiment
influencer_sentiment_data = []

for node in top_influencer_nodes:
    user_comm_id = partition.get(node)
    user_sentiment_label = user_sentiment_map.get(node, 'unknown')

    # Get dominant community sentiment from your filtered stats
    comm_row = filtered_stats_df[filtered_stats_df['Community ID'] == user_comm_id]
    comm_sentiment = comm_row['Dominant Sentiment'].values[0] if not comm_row.empty else 'unknown'

    influencer_sentiment_data.append({
        'Influencer': node,
        'Community ID': user_comm_id,
        'Influencer Sentiment': user_sentiment_label,
        'Community Sentiment': comm_sentiment,
        'Matches Community?': user_sentiment_label == comm_sentiment
    })

influencer_sentiment_df = pd.DataFrame(influencer_sentiment_data)
display(influencer_sentiment_df)

In [ ]:
def compare_sentiments(influencer_s, community_s):
    if influencer_s == community_s:
        return 'Matches'
    elif influencer_s == 'positive' and community_s != 'positive':
        return 'More Positive'
    elif influencer_s == 'negative' and community_s != 'negative':
        return 'More Negative'
    else:
        return 'Other'

influencer_sentiment_df['Sentiment Shift'] = influencer_sentiment_df.apply(
    lambda row: compare_sentiments(row['Influencer Sentiment'], row['Community Sentiment']), axis=1
)

In [ ]:
# See if the influencer's sentiment matches its community's
sentiment_counts = influencer_sentiment_df['Sentiment Shift'].value_counts()

sentiment_shift_colors = {
    'Matches': '#76c4ff',
    'More Positive': '#0fd129',
    'More Negative': '#f52e6a',
    'Other': '#fcca26'
}

plt.figure(figsize=(4, 4))
plt.pie(sentiment_counts, labels=sentiment_counts.index, autopct='%1.1f%%', startangle=140, colors=[sentiment_shift_colors[label] for label in sentiment_counts.index])

plt.title("Average Hubs Sentiment vs Community Sentiment")
plt.tight_layout()
plt.show()

It is generally expected that top influencers' sentiment (hubs) affects the sentiment of the whole community. However, here we see that a big group of users is also more negative than its community average.

## **SCC vs community sentiment**


In [ ]:
# Get biggest community
biggest_community_id = filtered_stats_df.sort_values('Size', ascending=False).iloc[0]['Community ID']
print(f"Biggest community ID: {biggest_community_id}")

# Extract users of biggest community
biggest_community_users = [u for u in partition if partition[u] == biggest_community_id]

# Build subgraph
G_biggest = G.subgraph(biggest_community_users).copy()
print(f"Number of users (nodes) in biggest community: {G_biggest.number_of_nodes()}")

In [ ]:
# Find connected components
components = list(nx.strongly_connected_components(G_biggest))
print(f"Found {len(components)} strongly connected components in biggest community")

In [ ]:
# Build user, withe the dominant sentiment map
user_sentiment_map = comments_df.groupby('author')['sentiment'].agg(lambda x: Counter(x).most_common(1)[0][0]).to_dict()

# For each component, compute sentiment counts
component_sentiment_counts = []

for idx, component in enumerate(components):
    sentiments = [user_sentiment_map.get(user, 'unknown') for user in component]
    counts = Counter(sentiments)

    component_sentiment_counts.append({
        'Component': f'Component {idx+1}',
        'Positive': counts.get('positive', 0),
        'Negative': counts.get('negative', 0),
        'Neutral': counts.get('neutral', 0),
    })

component_sentiment_df = pd.DataFrame(component_sentiment_counts)

All SCCs have less than 10 members each, except for one which has about 300 memebers. For simplicity and significance, we only analyse the biggest.

In [ ]:
#Find strongly connected components
components = list(nx.strongly_connected_components(G_biggest))
print(f"There are {len(components)} strongly connected components in biggest community")

#Sort components by size (largest first)
components_sorted = sorted(components, key=lambda x: len(x), reverse=True)

#Select the biggest component
biggest_component = components_sorted[0]
print(f"\nThe biggest strongly connected component groups {len(biggest_component)} users\n")

#Build sentiment count for this component
sentiments = [user_sentiment_map.get(user, 'unknown') for user in biggest_component]
counts = Counter(sentiments)

biggest_component_df = pd.DataFrame({
    'Sentiment': ['Positive', 'Negative', 'Neutral'],
    'Count': [counts.get('positive', 0), counts.get('negative', 0), counts.get('neutral', 0)]
})

custom_colors = ['#66CC00', '#FF0000', '#66B2FF']
plt.figure(figsize=(4,4))
plt.pie(
    biggest_component_df['Count'],
    labels=biggest_component_df['Sentiment'],
    autopct='%1.1f%%',
    startangle=140,
    colors=custom_colors
)

plt.title("Sentiment Distribution in Biggest Strongly Connected Component")
plt.tight_layout()
plt.show()